In [ ]:
!pip install numpy xarray matplotlib cartopy seaborn netCDF4


In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors


1. Download the latest VIIRS fire data from [NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/active_fire/).
2. Upload the `.txt` file using the left "Files" panel in Colab or update the `read_csv` path below.


In [ ]:
# Replace path with your uploaded file if needed
fires = pd.read_csv("/content/VIIRSNDE_global2024312.v1.0.txt")


In [ ]:
coverage = [-180.0, -90.0, 180.0, 90.0]
grid_size = 1.0
num_points_x = int((coverage[2] - coverage[0]) / grid_size)
num_points_y = int((coverage[3] - coverage[1]) / grid_size)

nx = complex(0, num_points_x)
ny = complex(0, num_points_y)
Xnew, Ynew = np.mgrid[coverage[0]:coverage[2]:nx, coverage[1]:coverage[3]:ny]

fire_count = np.zeros([num_points_x, num_points_y])
for i, lon in enumerate(fires['Lon']):
    lat = fires['Lat'][i]
    adjlat = (lat + 90) / grid_size
    adjlon = (lon + 180) / grid_size
    latbin = int(adjlat)
    lonbin = int(adjlon)
    fire_count[lonbin, latbin] += 1

fire_count[fire_count == 0] = np.nan


In [ ]:
cmap = plt.cm.get_cmap("hot")
norm = mcolors.Normalize(vmin=0, vmax=40)


In [ ]:
fig, ax = plt.subplots(figsize=[15, 15], subplot_kw={'projection': ccrs.Orthographic(central_longitude=90.0, central_latitude=0.0)})

ax.add_feature(cfeature.LAND, edgecolor='black')
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.LAKES, alpha=0.5)
ax.add_feature(cfeature.RIVERS)

sc = ax.scatter(fires['Lon'], fires['Lat'], s=1, transform=ccrs.PlateCarree(), color='blue', alpha=0.5, label='Fire Events')
pcm = ax.pcolormesh(Xnew, Ynew, fire_count, cmap=cmap, norm=norm, transform=ccrs.PlateCarree(), alpha=0.6)

cbar = plt.colorbar(pcm, ax=ax, orientation='horizontal', pad=0.05)
cbar.set_label('Fire Counts')

ax.set_global()
ax.coastlines()
plt.title('Global Fire Events with Binned Counts (1-degree grid)')
plt.savefig('global_fire_events.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()
